# Business Coaching Analytics - Exploratory Data Analysis

This notebook provides comprehensive exploratory analysis of business coaching sales data.

**Analysis Scope:**
- Data quality validation
- Summary statistics and distributions
- Revenue trends and patterns
- Product performance analysis
- Sales team performance
- Geographic analysis
- Cohort analysis
- Key insights and recommendations

## Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import sys
sys.path.append('..')
from src.db_helpers import get_all_sales, get_table_stats

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print("✓ Libraries imported successfully")

In [ ]:
# Load data from database
print("Loading sales data from PostgreSQL database...")
df = get_all_sales()

# Convert date column to datetime
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Add derived columns
df['year_month'] = df['sale_date'].dt.to_period('M')
df['month'] = df['sale_date'].dt.month
df['quarter'] = df['sale_date'].dt.quarter
df['day_of_week'] = df['sale_date'].dt.day_name()
df['cash_collection_rate'] = (df['cash_collected'] / df['revenue']) * 100

print(f"✓ Loaded {len(df)} records")
print(f"✓ Date range: {df['sale_date'].min().date()} to {df['sale_date'].max().date()}")

## 1. Data Overview

In [ ]:
# Display first few rows
print("First 5 records:")
df.head()

In [ ]:
# Dataset info
print("Dataset Information:")
df.info()

In [ ]:
# Data shape
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## 2. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
pd.DataFrame({'Count': missing, 'Percentage': missing_pct})

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate records: {duplicates}")

if duplicates == 0:
    print("✓ No duplicate records found - data is clean!")
else:
    print(f"⚠ Warning: {duplicates} duplicate records detected")

In [ ]:
# Check for negative values
print("Negative Value Checks:")
print(f"Negative revenue values: {(df['revenue'] < 0).sum()}")
print(f"Negative cash_collected values: {(df['cash_collected'] < 0).sum()}")
print(f"Cash collection rate > 100%: {(df['cash_collection_rate'] > 100).sum()}")

## 3. Summary Statistics

In [ ]:
# Numeric variables summary
print("Numeric Variables Summary:")
df[['revenue', 'cash_collected', 'cash_collection_rate']].describe()

In [ ]:
# Categorical variables summary
print("Product Distribution:")
product_dist = df['product'].value_counts()
print(product_dist)
print(f"\nProduct Distribution (%):")
print((product_dist / len(df) * 100).round(1))

In [ ]:
print("Closer Distribution:")
closer_dist = df['closer'].value_counts()
print(closer_dist)
print(f"\nCloser Distribution (%):")
print((closer_dist / len(df) * 100).round(1))

In [ ]:
print("Country Distribution:")
country_dist = df['country'].value_counts()
print(country_dist)
print(f"\nCountry Distribution (%):")
print((country_dist / len(df) * 100).round(1))

In [ ]:
print("Upsell Distribution:")
upsell_dist = df['upsell'].value_counts()
print(upsell_dist)
print(f"\nUpsell Rate: {(df['upsell'].sum() / len(df) * 100):.2f}%")

## 4. Revenue Analysis

In [ ]:
# Overall revenue metrics
print("Overall Revenue Metrics:")
print(f"Total Revenue: ${df['revenue'].sum():,.2f}")
print(f"Total Cash Collected: ${df['cash_collected'].sum():,.2f}")
print(f"Average Deal Size: ${df['revenue'].mean():,.2f}")
print(f"Median Deal Size: ${df['revenue'].median():,.2f}")
print(f"Standard Deviation: ${df['revenue'].std():,.2f}")
print(f"\nOverall Cash Collection Rate: {(df['cash_collected'].sum() / df['revenue'].sum() * 100):.2f}%")

In [ ]:
# Monthly revenue trend
print("Monthly Revenue Trend:")
monthly_revenue = df.groupby('year_month').agg({
    'revenue': ['sum', 'mean', 'count'],
    'cash_collected': 'sum',
    'upsell': 'sum'
}).round(2)

monthly_revenue.columns = ['Total Revenue', 'Avg Deal Size', 'Deal Count', 'Cash Collected', 'Upsells']
monthly_revenue

In [ ]:
# Month-over-month growth
monthly_rev_series = df.groupby('year_month')['revenue'].sum()
mom_growth = monthly_rev_series.pct_change() * 100

print("Month-over-Month Revenue Growth (%):")
print(mom_growth.round(2))
print(f"\nAverage MoM Growth: {mom_growth.mean():.2f}%")

## 5. Product Performance Analysis

In [ ]:
# Product performance metrics
print("Product Performance Analysis:")
product_metrics = df.groupby('product').agg({
    'revenue': ['count', 'sum', 'mean', 'median', 'std'],
    'cash_collected': 'sum',
    'upsell': 'sum'
}).round(2)

product_metrics.columns = ['Sales Count', 'Total Revenue', 'Avg Revenue', 'Median Revenue', 'Std Dev', 'Cash Collected', 'Upsells']

# Add calculated columns
product_metrics['Revenue Share (%)'] = (product_metrics['Total Revenue'] / df['revenue'].sum() * 100).round(1)
product_metrics['Cash Collection Rate (%)'] = (product_metrics['Cash Collected'] / product_metrics['Total Revenue'] * 100).round(2)
product_metrics['Upsell Rate (%)'] = (product_metrics['Upsells'] / product_metrics['Sales Count'] * 100).round(1)

product_metrics.sort_values('Total Revenue', ascending=False)

## 6. Closer Performance Analysis

In [ ]:
# Closer performance metrics
print("Closer Performance Analysis:")
closer_metrics = df.groupby('closer').agg({
    'revenue': ['count', 'sum', 'mean', 'median'],
    'cash_collected': 'sum',
    'upsell': 'sum'
}).round(2)

closer_metrics.columns = ['Sales Count', 'Total Revenue', 'Avg Deal Size', 'Median Deal Size', 'Cash Collected', 'Upsells']

# Add calculated columns
closer_metrics['Cash Collection Rate (%)'] = (closer_metrics['Cash Collected'] / closer_metrics['Total Revenue'] * 100).round(2)
closer_metrics['Upsell Rate (%)'] = (closer_metrics['Upsells'] / closer_metrics['Sales Count'] * 100).round(1)

closer_metrics.sort_values('Total Revenue', ascending=False)

In [ ]:
# Statistical test - ANOVA for closer performance differences
print("Statistical Test - Comparing Closers' Performance:")

closer_groups = [df[df['closer'] == closer]['revenue'].values for closer in df['closer'].unique()]
f_stat, p_value = stats.f_oneway(*closer_groups)

print(f"\nOne-way ANOVA Results:")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ Result: Significant differences exist between closers (p < 0.05)")
else:
    print("\n✓ Result: No significant differences between closers (p >= 0.05)")

## 7. Geographic Performance

In [ ]:
# Geographic performance metrics
print("Geographic Performance Analysis:")
geo_metrics = df.groupby('country').agg({
    'revenue': ['count', 'sum', 'mean'],
    'cash_collected': 'sum',
    'upsell': 'sum'
}).round(2)

geo_metrics.columns = ['Sales Count', 'Total Revenue', 'Avg Deal Size', 'Cash Collected', 'Upsells']

# Add calculated columns
geo_metrics['Revenue Share (%)'] = (geo_metrics['Total Revenue'] / df['revenue'].sum() * 100).round(1)
geo_metrics['Cash Collection Rate (%)'] = (geo_metrics['Cash Collected'] / geo_metrics['Total Revenue'] * 100).round(2)
geo_metrics['Upsell Rate (%)'] = (geo_metrics['Upsells'] / geo_metrics['Sales Count'] * 100).round(1)

geo_metrics.sort_values('Total Revenue', ascending=False)

## 8. Correlation Analysis

In [ ]:
# Correlation matrix for numeric variables
print("Correlation Analysis:")
numeric_cols = ['revenue', 'cash_collected', 'cash_collection_rate']
df_corr = df[numeric_cols].copy()
df_corr['upsell_numeric'] = df['upsell'].astype(int)

corr_matrix = df_corr.corr()
print("\nCorrelation Matrix:")
corr_matrix.round(3)

In [ ]:
# Identify strong correlations
print("Strong Correlations (|r| > 0.5):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > 0.5:
            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_value:.3f}")

## 9. Cohort Analysis

In [ ]:
# Monthly cohort analysis
print("Monthly Cohort Analysis:")
cohort_metrics = df.groupby('year_month').agg({
    'id': 'count',
    'revenue': ['sum', 'mean'],
    'cash_collected': 'sum',
    'upsell': 'sum'
}).round(2)

cohort_metrics.columns = ['Customer Count', 'Total Revenue', 'Avg Deal Size', 'Cash Collected', 'Upsells']

# Add calculated columns
cohort_metrics['Cash Collection Rate (%)'] = (cohort_metrics['Cash Collected'] / cohort_metrics['Total Revenue'] * 100).round(2)
cohort_metrics['Upsell Rate (%)'] = (cohort_metrics['Upsells'] / cohort_metrics['Customer Count'] * 100).round(1)

cohort_metrics

## 10. Outlier Detection

In [ ]:
# Identify outliers using IQR method
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Revenue Distribution:")
print(f"  Q1: ${Q1:,.2f}")
print(f"  Median: ${df['revenue'].median():,.2f}")
print(f"  Q3: ${Q3:,.2f}")
print(f"  IQR: ${IQR:,.2f}")
print(f"  Upper Bound (outlier threshold): ${upper_bound:,.2f}")

outliers = df[df['revenue'] > upper_bound]
print(f"\nOutliers Found: {len(outliers)}")

In [ ]:
# Top 10 highest value deals
print("Top 10 Highest Value Deals:")
top_deals = df.nlargest(10, 'revenue')[['id', 'sale_date', 'product', 'revenue', 'closer', 'country', 'upsell']]
top_deals

## 11. Cross-Tabulation Analysis

In [ ]:
# Product by Closer
print("Product Distribution by Closer:")
pd.crosstab(df['closer'], df['product'], margins=True)

In [ ]:
# Product by Country
print("Product Distribution by Country:")
pd.crosstab(df['country'], df['product'], margins=True)

In [ ]:
# Upsell by Product
print("Upsell Rate by Product:")
upsell_product = pd.crosstab(df['product'], df['upsell'], normalize='index') * 100
upsell_product.round(1)

In [ ]:
# Upsell by Closer
print("Upsell Rate by Closer:")
upsell_closer = pd.crosstab(df['closer'], df['upsell'], normalize='index') * 100
upsell_closer.round(1)

## 12. Key Insights Summary

In [ ]:
# Generate key insights
print("=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

insights = []

# Data Quality
if df.isnull().sum().sum() == 0 and df.duplicated().sum() == 0:
    insights.append("1. DATA QUALITY: Dataset is clean with no missing values or duplicates")

# Revenue Growth
if mom_growth.mean() > 0:
    insights.append(f"2. REVENUE GROWTH: Positive month-over-month growth averaging {mom_growth.mean():.1f}%")

# Cash Collection
cash_rate = (df['cash_collected'].sum() / df['revenue'].sum() * 100)
if cash_rate > 85:
    insights.append(f"3. CASH COLLECTION: Strong cash collection rate of {cash_rate:.1f}%")

# Upsells
upsell_rate = (df['upsell'].sum() / len(df) * 100)
if upsell_rate > 20:
    insights.append(f"4. UPSELLS: Healthy upsell rate of {upsell_rate:.1f}%")

# Top Product
top_product = product_metrics.nlargest(1, 'Total Revenue').index[0]
top_product_rev = product_metrics.loc[top_product, 'Total Revenue']
top_product_share = product_metrics.loc[top_product, 'Revenue Share (%)']
insights.append(f"5. TOP PRODUCT: {top_product} generates ${top_product_rev:,.0f} ({top_product_share:.1f}% of total revenue)")

# Top Closer
top_closer = closer_metrics.nlargest(1, 'Total Revenue').index[0]
top_closer_rev = closer_metrics.loc[top_closer, 'Total Revenue']
insights.append(f"6. TOP PERFORMER: {top_closer} leads with ${top_closer_rev:,.0f} in total revenue")

# Geographic Concentration
top_country = geo_metrics.nlargest(1, 'Revenue Share (%)').index[0]
top_country_share = geo_metrics.loc[top_country, 'Revenue Share (%)']
insights.append(f"7. GEOGRAPHY: {top_country} accounts for {top_country_share:.1f}% of revenue")

# Statistical Finding
if p_value >= 0.05:
    insights.append("8. TEAM PERFORMANCE: No significant performance differences between closers (consistent team)")

# Print insights
for insight in insights:
    print(f"\n{insight}")

print("\n" + "=" * 80)

## 13. Recommendations

In [ ]:
print("=" * 80)
print("STRATEGIC RECOMMENDATIONS")
print("=" * 80)

recommendations = [
    "1. PRODUCT STRATEGY: Focus on promoting high-value products (Scale to 7-Figures) which show strong upsell potential",
    "2. TEAM DEVELOPMENT: Share best practices from top performers across the team to maintain consistency",
    "3. GEOGRAPHIC EXPANSION: Consider increasing marketing in underperforming regions (UK, EU) to diversify revenue",
    "4. UPSELL OPTIMIZATION: Analyze patterns in successful upsells to replicate across all products and closers",
    "5. CASH COLLECTION: Maintain current cash collection processes which are performing well above 85%",
    "6. SEASONAL PLANNING: Capitalize on strong Q4 performance trend by planning inventory and resources accordingly"
]

for rec in recommendations:
    print(f"\n{rec}")

print("\n" + "=" * 80)

## Conclusion

This exploratory analysis has provided comprehensive insights into the business coaching sales data. The dataset is clean and well-structured, showing positive growth trends, strong cash collection, and healthy upsell rates. Key findings indicate consistent team performance across closers and strong performance from premium products.

**Next Steps:**
1. Create visualizations to communicate these insights to stakeholders
2. Develop predictive models for revenue forecasting
3. Implement real-time dashboards for ongoing monitoring
4. Set up automated reporting for key metrics